# L14 — Priority Queues and Reneging

**Module**: M05 | **Chapter**: 7 | **Lecture**: L14

## Learning Objectives
By the end of this notebook you will be able to:
1. Implement head-of-line (non-preemptive) priority using `simpy.PriorityResource`.
2. Model customer reneging (abandonment after patience expires) using SimPy's timeout-race pattern.
3. Quantify the trade-off between high-priority and low-priority customer wait times.
4. Compute the theoretical priority queue formulas and verify against simulation.

---
> **Think → Trace → Code → Experiment → Interpret → Communicate**
---

In [ ]:
import simpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t as t_dist

## 1. Priority Queue Theory

In a **non-preemptive M/M/1 priority queue** with k priority classes:

- Each class i has arrival rate λᵢ and uses the same server (μ)
- Total load: ρ = (λ₁+λ₂+...+λₖ)/μ < 1
- Mean wait in queue for class i:

$$W_q^{(i)} = \frac{W_q^{(0)}}{(1 - \sigma_{i-1})(1 - \sigma_i)}$$

where $\sigma_i = \sum_{j=1}^{i} \lambda_j/\mu$ (partial load through class i) and $W_q^{(0)} = \rho/\mu / (1-\rho)^2 / \mu$ is the FCFS baseline.

**Key result**: priority speeds up high-priority classes at the cost of slowing low-priority classes. Mean waiting time weighted by class mix is *identical* to FCFS.

In [ ]:
def priority_wq_theory(lam_list: list, mu: float) -> list:
    """Non-preemptive priority queue mean waits per class."""
    rho_total = sum(lam_list) / mu
    assert rho_total < 1.0

    # W0 = mean residual service time / (1-rho)^2 ... use standard formula
    # For M/M/1 priority: W_q^(0) = rho/(mu*(1-rho)) [the FCFS Wq]
    Wq_fcfs = rho_total / (mu * (1 - rho_total))

    results = []
    sigma_prev = 0.0
    for i, lam_i in enumerate(lam_list):
        sigma_i = sigma_prev + lam_i / mu
        Wq_i = Wq_fcfs / ((1 - sigma_prev) * (1 - sigma_i))
        results.append({'class': i+1, 'lam': lam_i, 'rho': lam_i/mu,
                        'Wq_priority': Wq_i, 'Wq_fcfs': Wq_fcfs})
        sigma_prev = sigma_i
    return results


# Two priority classes: urgent (priority 1) and routine (priority 2)
lam1, lam2, mu = 2.0, 3.0, 7.0
theory = priority_wq_theory([lam1, lam2], mu)

print(f"Non-preemptive priority queue (λ₁={lam1}, λ₂={lam2}, μ={mu}, ρ={sum([lam1,lam2])/mu:.3f})")
print(f"FCFS baseline Wq = {theory[0]['Wq_fcfs']:.4f}")
print()
for r in theory:
    improvement = (r['Wq_fcfs'] - r['Wq_priority']) / r['Wq_fcfs'] * 100
    print(f"  Class {r['class']} (λ={r['lam']}): Wq = {r['Wq_priority']:.4f}  "
          f"({'faster' if improvement>0 else 'slower'} than FCFS by {abs(improvement):.1f}%)")

## 2. SimPy `PriorityResource`: Non-Preemptive Priority

In [ ]:
def priority_queue_sim(lam_list: list, mu: float,
                       sim_time: float = 50_000, seed: int = 0) -> pd.DataFrame:
    """
    Non-preemptive priority queue simulation.
    Class 1 has highest priority (lowest numerical value in simpy).
    """
    rng = np.random.default_rng(seed)
    env = simpy.Environment()
    # PriorityResource: lower priority number = higher precedence
    server = simpy.PriorityResource(env, capacity=1)
    records = []

    def customer(priority: int):
        arrival = env.now
        # priority=1 is served before priority=2
        with server.request(priority=priority) as req:
            yield req
            wait = env.now - arrival
            svc  = rng.exponential(1.0 / mu)
            yield env.timeout(svc)
        records.append({'class': priority, 'wait': wait, 'sojourn': wait + svc})

    def arrivals(priority: int, lam: float):
        while True:
            yield env.timeout(rng.exponential(1.0 / lam))
            env.process(customer(priority))

    for i, lam in enumerate(lam_list, start=1):
        env.process(arrivals(i, lam))

    env.run(until=sim_time)
    return pd.DataFrame(records)


sim_df = priority_queue_sim([lam1, lam2], mu, sim_time=20_000, seed=42)

print("Priority queue simulation results:")
print(f"{'Class':>6s}  {'n':>8s}  {'Wq_sim':>10s}  {'Wq_theory':>12s}  {'Err %':>8s}")
print('-' * 55)
for cls, grp in sim_df.groupby('class'):
    wq_sim    = grp['wait'].mean()
    wq_theory = theory[cls-1]['Wq_priority']
    err       = abs(wq_sim - wq_theory) / wq_theory * 100
    print(f"{cls:>6d}  {len(grp):>8d}  {wq_sim:>10.4f}  {wq_theory:>12.4f}  {err:>7.2f}%")

In [ ]:
# Compare priority vs FCFS distributions
fcfs_df = priority_queue_sim([lam1, lam2], mu, sim_time=20_000, seed=42)
# For FCFS, assign all class=1 (same priority = FCFS order)
fcfs_df['class'] = 1   # ignore class distinction

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, grp_label in zip(axes, ['Priority 1 (urgent)', 'Priority 2 (routine)']):
    cls_id = 1 if 'urgent' in grp_label else 2
    pq_waits  = sim_df[sim_df['class']==cls_id]['wait'].values * mu  # normalise
    ax.hist(pq_waits, bins=50, density=True, alpha=0.7, color='steelblue', label='Priority queue')
    ax.axvline(pq_waits.mean(), color='steelblue', lw=2,
               label=f'Mean={pq_waits.mean():.2f}')
    ax.set_xlabel('Normalised wait μWq')
    ax.set_ylabel('Density')
    ax.set_title(grp_label)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Priority queue: class 1 benefits at expense of class 2', y=1.02)
plt.tight_layout()
plt.show()

## 3. Reneging: The Timeout-Race Pattern

**Reneging** (abandonment): a customer waits up to a patience time `T_max`, then leaves if not yet served.

SimPy idiom: race a `Resource.request()` against a `timeout(T_max)`. Whichever resolves first wins.

In [ ]:
def reneging_sim(lam: float, mu: float,
                 patience_mean: float,        # mean patience time (Exp)
                 sim_time: float = 50_000,
                 seed: int = 0) -> dict:
    """
    M/M/1 queue with exponentially distributed patience (reneging).
    Returns statistics for served and reneged customers separately.
    """
    rng = np.random.default_rng(seed)
    env = simpy.Environment()
    server = simpy.Resource(env, capacity=1)

    served  = []   # sojourn times of customers who completed service
    reneged = []   # wait times of customers who gave up

    def customer():
        arrival  = env.now
        patience = rng.exponential(patience_mean)  # how long willing to wait

        req = server.request()
        # Race: request vs. patience timeout
        result = yield req | env.timeout(patience)

        if req in result:  # won the race — server granted before patience expired
            wait = env.now - arrival
            svc  = rng.exponential(1.0 / mu)
            yield env.timeout(svc)
            server.release(req)  # release resource after service
            served.append({'wait': wait, 'sojourn': wait + svc})
        else:              # patience expired — renege
            req.cancel()   # remove from queue
            reneged.append({'wait': patience})

    def arrivals():
        while True:
            yield env.timeout(rng.exponential(1.0 / lam))
            env.process(customer())

    env.process(arrivals())
    env.run(until=sim_time)

    n_total   = len(served) + len(reneged)
    renege_rate = len(reneged) / n_total if n_total > 0 else 0
    return {
        'renege_rate': renege_rate,
        'Wq_served':   np.mean([r['wait'] for r in served])  if served  else 0,
        'Wq_reneged':  np.mean([r['wait'] for r in reneged]) if reneged else 0,
        'n_served':    len(served),
        'n_reneged':   len(reneged),
        'throughput':  len(served) / sim_time,
    }


# High-load scenario (ρ=0.95 without reneging)
lam_r, mu_r = 9.5, 10.0
print(f"Base utilisation (no reneging): ρ = {lam_r/mu_r:.2f}")
print()
print(f"{'patience_mean':>14s}  {'renege_rate':>12s}  {'Wq_served':>10s}  {'throughput':>12s}")
print('-' * 55)
for pm in [0.5, 1.0, 2.0, 5.0, float('inf')]:
    if pm == float('inf'):
        r = reneging_sim(lam_r, mu_r, patience_mean=1e9, sim_time=10_000, seed=0)
        label = '∞ (no reneging)'
    else:
        r = reneging_sim(lam_r, mu_r, patience_mean=pm, sim_time=10_000, seed=0)
        label = f'{pm:.1f}'
    print(f"{label:>14s}  {r['renege_rate']:>12.3f}  {r['Wq_served']:>10.4f}  {r['throughput']:>12.4f}")

## 4. Priority + Reneging: A Realistic Triage Model

Combine both features: urgent patients (class 1) never renege; routine patients (class 2) renege after Exp(20 min).

In [ ]:
def triage_sim(lam_urgent: float, lam_routine: float, mu: float,
               patience_routine: float,
               sim_time: float = 50_000, seed: int = 0) -> dict:
    rng = np.random.default_rng(seed)
    env = simpy.Environment()
    server = simpy.PriorityResource(env, capacity=1)
    urgent_waits, routine_waits, routine_reneged = [], [], []

    def urgent_patient():
        arrival = env.now
        with server.request(priority=1) as req:
            yield req
            wait = env.now - arrival
            yield env.timeout(rng.exponential(1.0 / mu))
        urgent_waits.append(wait)

    def routine_patient():
        arrival  = env.now
        patience = rng.exponential(patience_routine)
        req = server.request(priority=2)
        result = yield req | env.timeout(patience)
        if req in result:
            wait = env.now - arrival
            yield env.timeout(rng.exponential(1.0 / mu))
            server.release(req)  # release resource after service
            routine_waits.append(wait)
        else:
            req.cancel()
            routine_reneged.append(patience)

    def arrivals(gen_fn, lam):
        while True:
            yield env.timeout(rng.exponential(1.0 / lam))
            env.process(gen_fn())

    env.process(arrivals(urgent_patient,  lam_urgent))
    env.process(arrivals(routine_patient, lam_routine))
    env.run(until=sim_time)

    n_routine_total = len(routine_waits) + len(routine_reneged)
    return {
        'Wq_urgent':         np.mean(urgent_waits),
        'Wq_routine_served': np.mean(routine_waits) if routine_waits else 0,
        'renege_rate_routine': len(routine_reneged) / n_routine_total if n_routine_total else 0,
        'n_urgent':  len(urgent_waits),
        'n_routine': len(routine_waits),
        'n_reneged': len(routine_reneged),
    }


r = triage_sim(lam_urgent=2.0, lam_routine=5.0, mu=8.0,
               patience_routine=1/6,  # 10 min in hour units
               seed=42)

print("Priority + Reneging triage model (T=200,000 hr):")
print(f"  Urgent mean wait:         {r['Wq_urgent']*60:.2f} min")
print(f"  Routine mean wait (served):{r['Wq_routine_served']*60:.2f} min")
print(f"  Routine abandonment rate: {r['renege_rate_routine']*100:.1f}%")
print(f"  Urgent served: {r['n_urgent']:,}   Routine served: {r['n_routine']:,}   Reneged: {r['n_reneged']:,}")

---
## Try It Yourself

1. **Preemptive priority**: Change `PriorityResource` to `PreemptiveResource`. An urgent patient who arrives while a routine patient is being served will *interrupt* the service. Count interrupted patients and measure the extra delay caused. Compare preemptive vs. non-preemptive mean waits for both classes.

2. **Service-level agreement**: Management wants urgent patients to wait < 2 minutes with probability ≥ 0.99. Use your `triage_sim` to find the minimum server capacity (μ) that achieves this, given λ_urgent=3/hr and λ_routine=7/hr and patience_routine=15 min.

3. **Balking**: Instead of reneging (leaving after waiting), implement **balking** (refusing to join a long queue). Add a parameter: a customer joins the queue only if fewer than K customers are already waiting. Simulate with K=5 and compare throughput, renege rate, and Wq to the reneging model.